## 1. 미션 목표
- 이번 미션의 목표는 Hugging Face `transformers` 라이브러리를 활용해 한국어 문서 요약 파이프라인을 직접 구성해 보는 것이다.
- 단순히 모델을 실행하는 데서 끝나지 않고, 데이터 구조를 이해하고 전처리 기준을 정한 뒤 요약 결과를 해석하는 과정까지 포함한다.
- 최종적으로는 원문, 정답 요약, 생성 요약을 함께 비교하여 모델이 실제로 어떤 수준의 요약을 수행하는지 확인한다.

In [1]:
# STEP1_ENV_SETUP
# 1) 현재 커널 기준으로 필요한 패키지 점검 및 설치
import importlib.util
import json
import os
import random
import re
import subprocess
import sys
import warnings
from pathlib import Path

REQUIRED_PACKAGES = {
    "datasets": "datasets",
    "transformers": "transformers",
    "accelerate": "accelerate>=1.1.0",
    "sentencepiece": "sentencepiece",
    "evaluate": "evaluate",
    "rouge_score": "rouge-score",
}

missing_packages = [
    spec for module_name, spec in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("설치 필요한 패키지:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

# 2) 재현성 고정 및 기본 라이브러리 import
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def find_data_dir() -> Path:
    candidates = [
        Path.cwd() / "content" / "summarization",
        Path.cwd() / "미션" / "content" / "summarization",
        Path.cwd().parent / "content" / "summarization",
        Path.cwd().parent / "미션" / "content" / "summarization",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("summarization 데이터 폴더를 찾지 못했습니다.")

DATA_DIR = find_data_dir()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"DATA_DIR: {DATA_DIR}")
print(f"DEVICE: {DEVICE}")
print(f"SEED: {SEED}")


c:\Users\amy\Desktop\sprint\sprint_ai07\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_DIR: c:\Users\amy\Desktop\sprint\sprint_ai07\미션\content\summarization
DEVICE: cuda
SEED: 42


## 2. 데이터 소개
- 사용 데이터는 AI HUB 문서요약 텍스트 데이터이며, `news`, `editorial`, `law` 세 가지 유형의 문서로 구성되어 있다.
- 각 유형은 학습용과 검증용 JSON 파일로 나뉘어 있으며, 문서 본문과 사람이 작성한 요약문이 함께 제공된다.
- 이번 실습에서는 전체 데이터를 한 번에 쓰기보다, 먼저 하나의 도메인을 선택해 안정적으로 파이프라인을 완성한 뒤 필요하면 확장하는 방향으로 진행한다.

In [2]:
# STEP2_DATASET_OVERVIEW
# 1) 데이터 파일 목록과 크기 확인
file_rows = []
for file_path in sorted(DATA_DIR.glob("*.json")):
    parts = file_path.stem.split("_")
    file_rows.append(
        {
            "file_name": file_path.name,
            "split": parts[0],
            "domain": parts[-1],
            "size_mb": round(file_path.stat().st_size / (1024 ** 2), 2),
        }
    )

file_df = pd.DataFrame(file_rows)
display(file_df)

SELECTED_DOMAIN = "law"
train_path = DATA_DIR / f"train_original_{SELECTED_DOMAIN}.json"
valid_path = DATA_DIR / f"valid_original_{SELECTED_DOMAIN}.json"

print(f"선택 도메인: {SELECTED_DOMAIN}")
print(f"train path: {train_path.name}")
print(f"valid path: {valid_path.name}")


,file_name,split,domain,size_mb
0,train_original_editorial.json,train,editorial,313.23
1,train_original_law.json,train,law,82.42
2,train_original_news.json,train,news,1108.73
3,valid_original_editorial.json,valid,editorial,35.33
4,valid_original_law.json,valid,law,8.51
5,valid_original_news.json,valid,news,139.97


선택 도메인: law
train path: train_original_law.json
valid path: valid_original_law.json


## 3. 데이터 로드
- 먼저 데이터 폴더 안의 JSON 파일 목록을 확인하고, 실험에 사용할 문서 유형과 train/valid 파일을 선택한다.
- 각 JSON 파일에서 문서 단위로 본문과 요약을 추출해, 모델 학습에 사용할 `text-summary` 쌍 형태로 정리한다.
- 이 단계에서는 샘플 몇 개를 직접 출력해 원문 구조가 중첩 리스트인지, 요약이 문자열 리스트인지 먼저 확인하는 과정이 중요하다.

In [3]:
# STEP3_JSON_SCHEMA_CHECK
# 1) 샘플 문서를 확인해 원문/요약 구조 파악
with open(valid_path, "r", encoding="utf-8") as f:
    valid_json = json.load(f)

sample_doc = valid_json["documents"][0]
sample_sentences = [
    sent_info["sentence"]
    for block in sample_doc["text"]
    for sent_info in block
]

print(f"검증 문서 수: {len(valid_json['documents'])}")
print(f"문서 키: {list(sample_doc.keys())}")
print(f"본문 문장 수: {len(sample_sentences)}")
print(f"제목: {sample_doc.get('title', '(제목 없음)')}")
print(f"요약 예시: {sample_doc['abstractive'][0]}")
print("원문 앞부분:")
print(" ".join(sample_sentences[:3])[:400] + "...")


검증 문서 수: 3004
문서 키: ['id', 'category', 'size', 'char_count', 'publish_date', 'title', 'text', 'annotator_id', 'document_quality_scores', 'extractive', 'abstractive']
본문 문장 수: 7
제목: 사도개설허가취소신청거부처분취소
요약 예시: 취소소송은 처분 등이 있다는 것을 안 때로부터 90일 이내에 제기하여야 하고, 행정처분에서의 허가에 붙은 기한이 부당하게 짧은 경우에는 이를 허가조건 존속기간으로 보아서 그 기한의 도래로 조건 개정을 고려한다고 해석할 수 있기에, 사도개설허가의 준공검사를 받지 못한 것은 사도개설허가 자체의 존속기간으로 볼 수 없다는 까닭으로 이것이 실효되는 것은 아니다.
원문 앞부분:
[1] 취소소송은 처분 등이 있음을 안 날부터 90일 이내에 제기하여야 하고, 처분 등이 있은 날부터 1년을 경과하면 제기하지 못하며( 행정소송법 제20조 제1항, 제2항), 청구취지를 변경하여 구 소가 취하되고 새로운 소가 제기된 것으로 변경되었을 때에 새로운 소에 대한 제소기간의 준수 등은 원칙적으로 소의 변경이 있은 때를 기준으로 하여야 한다....


## 4. 전처리 기준
- 본문은 여러 문장 조각으로 나뉘어 있으므로, 문장 순서를 유지한 채 하나의 입력 텍스트로 결합하는 기준을 정한다.
- 불필요한 공백, 줄바꿈, 과도한 기호는 최소한으로 정리하되, 문서 의미가 손상되지 않도록 과한 정제는 피한다.
- 또한 비어 있는 샘플, 지나치게 짧은 문서, 요약이 누락된 샘플이 있는지 점검하여 학습 데이터 품질을 확보한다.

In [4]:
# STEP4_PREPROCESS_AND_LOAD
# 1) 텍스트 정리 함수와 데이터프레임 생성 함수 정의
def normalize_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def join_document_text(text_blocks) -> str:
    sentences = []
    for block in text_blocks:
        for sent_info in block:
            sentence = normalize_text(sent_info.get("sentence", ""))
            if sentence:
                sentences.append(sentence)
    return " ".join(sentences)

def extract_summary(doc) -> str:
    abstractive = doc.get("abstractive", [])
    if isinstance(abstractive, list):
        summary = " ".join(
            normalize_text(item) for item in abstractive if normalize_text(item)
        )
    else:
        summary = normalize_text(abstractive)
    return summary

def load_split_dataframe(file_path: Path, split_name: str, sample_limit=None) -> pd.DataFrame:
    with open(file_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    rows = []
    documents = payload["documents"]
    if sample_limit is not None and sample_limit < len(documents):
        rng = random.Random(SEED if split_name == "train" else SEED + 1)
        documents = rng.sample(documents, sample_limit)
    elif sample_limit is not None:
        documents = documents[:sample_limit]
    for doc in documents:
        text = join_document_text(doc.get("text", []))
        summary = extract_summary(doc)
        if not text or not summary:
            continue
        rows.append(
            {
                "split": split_name,
                "document_id": doc.get("id"),
                "title": normalize_text(doc.get("title", "")),
                "category": doc.get("category", SELECTED_DOMAIN),
                "text": text,
                "summary": summary,
            }
        )

    df = pd.DataFrame(rows)
    df["text_len_chars"] = df["text"].str.len()
    df["summary_len_chars"] = df["summary"].str.len()
    df["text_len_words"] = df["text"].str.split().str.len()
    df["summary_len_words"] = df["summary"].str.split().str.len()
    return df

TRAIN_SAMPLE_LIMIT = 2000
VALID_SAMPLE_LIMIT = 400

train_df = load_split_dataframe(train_path, "train", TRAIN_SAMPLE_LIMIT)
valid_df = load_split_dataframe(valid_path, "valid", VALID_SAMPLE_LIMIT)

display(train_df.head(2))
print(f"train shape: {train_df.shape}")
print(f"valid shape: {valid_df.shape}")


,split,document_id,title,category,text,summary,text_len_chars,summary_len_chars,text_len_words,summary_len_words
0,train,203536,부가가치세부과처분취소,세무,가. 부가가치세법시행령 제74조의3 제3항이 일반과세자가 과세특례자로 변경된 경우에...,과세특례에 관한 규정의 적용요건은 법정되어 있고 그 적용 여부는 요건에 해당하는지의...,1839,255,417,59
1,train,111052,양도소득세등부과처분취소,세무,구 소득세법시행령(1989.8.1. 대통령령 제12767호로 개정되기 전의 것) 제...,구 소득세법시행령 제170조 소정의 법인과의 거래인지 여부는 당사자의 의사 등 실질...,469,213,101,47


train shape: (2000, 10)
valid shape: (400, 10)


## 5. 길이 분석
- 요약 모델은 입력 길이 제한의 영향을 크게 받기 때문에, 본문과 요약문의 길이 분포를 먼저 확인한다.
- 길이 분석 결과를 바탕으로 `max_input_length`, `max_target_length` 같은 토크나이징 기준을 정한다.
- 이 과정은 메모리 사용량과 학습 시간, 그리고 요약 품질에 직접 연결되므로 실험 설정의 근거를 제공한다.

In [5]:
# STEP5_LENGTH_ANALYSIS
# 1) 문자 수와 어절 수 기준 길이 분포 확인
length_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "samples": len(train_df),
            "text_char_mean": round(train_df["text_len_chars"].mean(), 1),
            "summary_char_mean": round(train_df["summary_len_chars"].mean(), 1),
            "text_word_mean": round(train_df["text_len_words"].mean(), 1),
            "summary_word_mean": round(train_df["summary_len_words"].mean(), 1),
        },
        {
            "split": "valid",
            "samples": len(valid_df),
            "text_char_mean": round(valid_df["text_len_chars"].mean(), 1),
            "summary_char_mean": round(valid_df["summary_len_chars"].mean(), 1),
            "text_word_mean": round(valid_df["text_len_words"].mean(), 1),
            "summary_word_mean": round(valid_df["summary_len_words"].mean(), 1),
        },
    ]
)
display(length_summary)

length_percentiles = pd.DataFrame(
    {
        "metric": ["text_len_chars", "summary_len_chars", "text_len_words", "summary_len_words"],
        "train_p50": [
            int(train_df["text_len_chars"].quantile(0.50)),
            int(train_df["summary_len_chars"].quantile(0.50)),
            int(train_df["text_len_words"].quantile(0.50)),
            int(train_df["summary_len_words"].quantile(0.50)),
        ],
        "train_p90": [
            int(train_df["text_len_chars"].quantile(0.90)),
            int(train_df["summary_len_chars"].quantile(0.90)),
            int(train_df["text_len_words"].quantile(0.90)),
            int(train_df["summary_len_words"].quantile(0.90)),
        ],
        "valid_p50": [
            int(valid_df["text_len_chars"].quantile(0.50)),
            int(valid_df["summary_len_chars"].quantile(0.50)),
            int(valid_df["text_len_words"].quantile(0.50)),
            int(valid_df["summary_len_words"].quantile(0.50)),
        ],
        "valid_p90": [
            int(valid_df["text_len_chars"].quantile(0.90)),
            int(valid_df["summary_len_chars"].quantile(0.90)),
            int(valid_df["text_len_words"].quantile(0.90)),
            int(valid_df["summary_len_words"].quantile(0.90)),
        ],
    }
)
display(length_percentiles)


,split,samples,text_char_mean,summary_char_mean,text_word_mean,summary_word_mean
0,train,2000,671.4,193.9,152.1,43.8
1,valid,400,525.4,191.7,119.3,43.2


,metric,train_p50,train_p90,valid_p50,valid_p90
0,text_len_chars,522,1261,417,930
1,summary_len_chars,184,289,181,285
2,text_len_words,120,287,93,212
3,summary_len_words,42,66,41,65


## 6. 모델 선택 이유
- 이번 실습에서는 한국어 문서 요약에 적합한 사전학습 모델을 선택하고, 그 이유를 명확히 설명하는 것이 중요하다.
- 선택 기준은 한국어 지원 여부, 요약 태스크 적합성, 현재 노트북 환경에서의 실행 가능성, 파인튜닝 난이도 등으로 정리할 수 있다.
- baseline 코드를 그대로 따르기보다, 내가 왜 이 모델을 선택했는지 실험 맥락과 함께 서술하는 방향으로 정리한다.

In [6]:
# STEP6_MODEL_AND_TOKENIZER
# 1) 사전학습 요약 모델과 토크나이저 준비
MODEL_NAME = "digit82/kobart-summarization"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def get_token_lengths(text_series: pd.Series, sample_size: int = 400):
    sampled_texts = text_series.iloc[: min(len(text_series), sample_size)].tolist()
    return [len(tokenizer(text, truncation=False)["input_ids"]) for text in sampled_texts]

train_input_token_lens = get_token_lengths(train_df["text"])
train_target_token_lens = get_token_lengths(train_df["summary"])

MAX_INPUT_LENGTH = min(512, max(384, int(np.percentile(train_input_token_lens, 95))))
MAX_TARGET_LENGTH = min(160, max(128, int(np.percentile(train_target_token_lens, 95))))
GEN_MAX_LENGTH = min(144, MAX_TARGET_LENGTH + 16)

print(f"MODEL_NAME: {MODEL_NAME}")
print(f"MAX_INPUT_LENGTH: {MAX_INPUT_LENGTH}")
print(f"MAX_TARGET_LENGTH: {MAX_TARGET_LENGTH}")
print(f"GEN_MAX_LENGTH: {GEN_MAX_LENGTH}")
print(pd.Series(train_input_token_lens).describe(percentiles=[0.5, 0.9, 0.95]))


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


MODEL_NAME: digit82/kobart-summarization
MAX_INPUT_LENGTH: 512
MAX_TARGET_LENGTH: 144
GEN_MAX_LENGTH: 144
count     400.000000
mean      277.352500
std       176.666979
min        51.000000
50%       214.000000
90%       530.200000
95%       665.400000
max      1207.000000
dtype: float64


## 7. 학습/추론
- 전처리된 데이터를 Hugging Face `Dataset` 형식으로 변환한 뒤 토크나이저를 적용해 모델 입력 형태로 만든다.
- 이후 학습용 하이퍼파라미터를 설정하고, 사전학습 모델을 데이터에 맞게 파인튜닝하거나 최소한 추론 파이프라인을 실행한다.
- 학습이 끝난 뒤에는 검증 데이터 일부를 사용해 실제 요약문을 생성하고, 모델이 어떤 방식으로 핵심 내용을 압축하는지 확인한다.

In [7]:
# STEP7_DATASET_AND_TOKENIZATION
# 1) Hugging Face Dataset 변환 및 토크나이징
train_dataset = Dataset.from_pandas(
    train_df[["document_id", "title", "text", "summary"]], preserve_index=False
)
valid_dataset = Dataset.from_pandas(
    valid_df[["document_id", "title", "text", "summary"]], preserve_index=False
)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
})

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

display(tokenized_datasets)
print(tokenized_datasets["train"][0].keys())


Map: 100%|██████████| 400/400 [00:00<00:00, 1818.52 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 400
    })
})

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [8]:
# STEP8_TRAINER_SETUP_AND_OPTIONAL_TRAINING
# 1) 모델, 데이터 콜레이터, Trainer 구성
import inspect
import transformers

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

OUTPUT_DIR = Path("outputs") / f"mission12_{SELECTED_DOMAIN}_kobart"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FOR_SUBMISSION = True
TRAIN_EPOCHS = 3
GRAD_ACCUM_STEPS = 2
RUN_TRAINING = TRAIN_FOR_SUBMISSION and torch.cuda.is_available()

arg_names = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters.keys()
training_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "num_train_epochs": TRAIN_EPOCHS,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 4,
    "learning_rate": 3e-5,
    "weight_decay": 0.02,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "save_strategy": "epoch",
    "logging_strategy": "steps",
    "logging_steps": 20,
    "predict_with_generate": True,
    "fp16": torch.cuda.is_available(),
    "report_to": "none",
    "save_total_limit": 2,
}

if "overwrite_output_dir" in arg_names:
    training_kwargs["overwrite_output_dir"] = True

if "evaluation_strategy" in arg_names:
    training_kwargs["evaluation_strategy"] = "epoch"
elif "eval_strategy" in arg_names:
    training_kwargs["eval_strategy"] = "epoch"

if "load_best_model_at_end" in arg_names:
    training_kwargs["load_best_model_at_end"] = True

if "metric_for_best_model" in arg_names:
    training_kwargs["metric_for_best_model"] = "eval_loss"

if "greater_is_better" in arg_names:
    training_kwargs["greater_is_better"] = False

if "generation_max_length" in arg_names:
    training_kwargs["generation_max_length"] = GEN_MAX_LENGTH

if "generation_num_beams" in arg_names:
    training_kwargs["generation_num_beams"] = 5

if "warmup_ratio" in arg_names:
    training_kwargs["warmup_ratio"] = 0.05

training_args = Seq2SeqTrainingArguments(**training_kwargs)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

eval_metrics = None

print(f"transformers version: {transformers.__version__}")
print(f"TRAIN_FOR_SUBMISSION: {TRAIN_FOR_SUBMISSION}")
print(f"RUN_TRAINING: {RUN_TRAINING}")
print(f"TRAIN_EPOCHS: {TRAIN_EPOCHS}")
print(f"GRAD_ACCUM_STEPS: {GRAD_ACCUM_STEPS}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

if RUN_TRAINING:
    train_result = trainer.train()
    eval_metrics = trainer.evaluate()
    trainer.save_model()
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(train_result.metrics)
    print(eval_metrics)
else:
    print("현재 환경에서는 기본 설정상 학습을 건너뜁니다.")
    print("GPU 환경에서 전체 학습을 진행하고 싶다면 TRAIN_FOR_SUBMISSION 값을 True로 두고 실행하세요.")


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 262/262 [00:00<00:00, 610.08it/s, Materializing param=model.shared.weight]                                  
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


transformers version: 5.2.0
TRAIN_FOR_SUBMISSION: True
RUN_TRAINING: True
TRAIN_EPOCHS: 3
GRAD_ACCUM_STEPS: 2
OUTPUT_DIR: outputs\mission12_law_kobart


Epoch,Training Loss,Validation Loss
1,3.020672,1.132175
2,2.028589,1.122644
3,1.753708,1.142288


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.41s/it]
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.26s/it]

{'train_runtime': 280.4646, 'train_samples_per_second': 21.393, 'train_steps_per_second': 5.348, 'total_flos': 1253618158141440.0, 'train_loss': 2.3565003255208334, 'epoch': 3.0}
{'eval_loss': 1.122644066810608, 'eval_runtime': 3.4415, 'eval_samples_per_second': 116.227, 'eval_steps_per_second': 29.057, 'epoch': 3.0}


## 8. 정성 평가
- 정성 평가는 숫자보다 실제 생성 결과를 읽으며 판단하는 단계로, 원문과 정답 요약, 생성 요약을 나란히 비교하는 방식이 가장 직관적이다.
- 이때 핵심 정보 보존 여부, 문장 자연스러움, 중복 표현, 중요한 사실 누락 여부를 중심으로 해석한다.
- 몇 개의 사례를 직접 분석하면 모델의 장점과 한계를 훨씬 설득력 있게 제시할 수 있다.

이번 정성 평가 결과를 보면, 학습된 모델은 법률 문서의 핵심 쟁점과 판시 요지를 어느 정도 따라가며 요약을 생성하는 모습을 보였다. 특히 세금, 행정처분, 계약 관련 사례처럼 핵심 법리가 비교적 분명한 문서에서는 정답 요약과 유사한 방향으로 중요한 내용을 압축하는 경향이 확인되었다.

다만 생성 결과를 자세히 살펴보면 여전히 원문 표현을 길게 따라가는 추출형 성향이 남아 있고, 일부 샘플에서는 문장이 길어지거나 어색하게 반복되는 부분도 나타났다. 즉, 전체 파이프라인은 안정적으로 작동했고 요약의 방향성도 적절했지만, 더 간결하고 자연스러운 요약문을 만들기 위해서는 데이터 규모 확대, 생성 파라미터 추가 조정, 또는 더 긴 학습이 필요하다고 해석할 수 있다.

In [9]:
# STEP9_QUALITATIVE_EVALUATION
# 1) 검증 샘플에 대한 생성 요약 확인
def generate_summary(text: str) -> str:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(DEVICE)
    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=GEN_MAX_LENGTH,
            min_length=max(40, int(MAX_TARGET_LENGTH * 0.55)),
            num_beams=5,
            no_repeat_ngram_size=3,
            length_penalty=1.2,
            repetition_penalty=1.1,
            early_stopping=True,
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

QUAL_EVAL_SAMPLES = 5
qualitative_results = valid_df[["document_id", "title", "text", "summary"]].sample(
    n=min(QUAL_EVAL_SAMPLES, len(valid_df)),
    random_state=SEED,
).reset_index(drop=True)
qualitative_results["generated_summary"] = qualitative_results["text"].apply(generate_summary)

for idx, row in qualitative_results.iterrows():
    print(f"[샘플 {idx + 1}]")
    print(f"제목: {row['title']}")
    print("원문 일부:")
    print(row["text"][:350] + "...")
    print(f"정답 요약: {row['summary']}")
    print(f"생성 요약: {row['generated_summary']}")
    print(f"정답 길이(문자 수): {len(row['summary'])} | 생성 길이(문자 수): {len(row['generated_summary'])}")
    print("-" * 100)


[샘플 1]
제목: 상속세부과처분취소
원문 일부:
[1] 세법상 가산세는 과세권의 행사 및 조세채권의 실현을 용이하게 하기 위하여 납세자가 정당한 이유 없이 법에 규정된 신고, 납세 등 각종 의무를 위반한 경우에 법이 정하는 바에 따라 부과되는 행정상 제재로서 그 의무의 이행을 납세의무자에게 기대하는 것이 무리인 사정이 있을 때 등 그 의무해태를 탓할 수 없는 정당한 사유가 있는 경우에는 이를 부과할 수 없다. [2] 상속세 신고가 납세의무자들이 아닌 유언집행자들에 의하여 행해졌고, 유언집행을 위하여 필요한 범위 내에서는 유언집행자의 상속재산에 대한 관리처분권이 상속인의 그것보다 우선할 뿐만 아니라 위 상속세 신고 당시 상속재산의 일부를 장학기금으로 출연하라는 망인의 ...
정답 요약: 세법상 가산세는 납세자가 정당한 이유 없이 각종 의무를 위반한 경우에 부과되는 행정상 제재로서 납세의무자에게 의무 이행을 기대하는 것이 무리인 사정이 있을 때 등 정당한 사유가 있는 경우에는 이를 부과할 수 없는데 위 사례 상속세 납세의무자들에게는 상속세 과소신고·납부를 탓할 수 없는 정당한 사유가 있다고 본 사례이다.
생성 요약: 상속세 신고가 납세의무자들이 아닌 유언집행자들에 의해 행해졌고 상속재산의 일부를 장학기금으로 출연하라는 망인의 유언의 효력이 미확정인 상태에 있었던 점 등에 비추어 상속세 납부자들에게 상속세 과소신고·납부를 탓할 수 없는 정당한 사유가 있다고 한 사례이다.    상속세 과세액에 포함시켜 상속세를 신고·납부할 것을 기대하는 것은 무리가 있으므로 상속세납세의무자들에게 상속세를 과소 신고,납부를 청구할 수 없다.  
정답 길이(문자 수): 179 | 생성 길이(문자 수): 231
----------------------------------------------------------------------------------------------------
[샘플 2]
제목: 손해배상(기)
원문 일부:
[1] 의료행위는 고도의 전문적 지식을 필요로 하는 

## 9. Optional 정량 평가
- 시간이 허용되면 ROUGE 같은 자동 평가 지표를 이용해 생성 요약과 정답 요약의 유사도를 수치로 비교할 수 있다.
- 다만 정량 평가는 설치 환경과 패키지 의존성의 영향을 받을 수 있으므로, 본 미션에서는 선택 과제로 두고 진행해도 무방하다.
- 점수는 참고 자료로 활용하되, 정성 평가와 함께 해석해야 실제 요약 품질을 더 정확하게 판단할 수 있다.

In [10]:
# STEP10_OPTIONAL_ROUGE_EVAL
# 1) evaluate가 준비되어 있으면 ROUGE 계산, 아니면 안내 메시지 출력
rouge_scores = None

try:
    import evaluate

    rouge = evaluate.load("rouge")
    rouge_scores = rouge.compute(
        predictions=qualitative_results["generated_summary"].tolist(),
        references=qualitative_results["summary"].tolist(),
    )
    rouge_df = pd.DataFrame([rouge_scores]).T.rename(columns={0: "score"})
    display(rouge_df)
except Exception as e:
    print("ROUGE 계산은 선택 사항입니다.")
    print("필요 시 `!pip install evaluate rouge-score` 실행 후 이 셀을 다시 실행하세요.")
    print(f"현재 상태: {e}")

# 2) 제출용 핵심 정보 요약
final_summary = pd.DataFrame(
    {
        "item": [
            "selected_domain",
            "train_samples",
            "valid_samples",
            "model_name",
            "max_input_length",
            "max_target_length",
            "train_epochs",
            "gradient_accumulation_steps",
            "run_training",
            "eval_loss",
        ],
        "value": [
            SELECTED_DOMAIN,
            len(train_df),
            len(valid_df),
            MODEL_NAME,
            MAX_INPUT_LENGTH,
            MAX_TARGET_LENGTH,
            TRAIN_EPOCHS,
            GRAD_ACCUM_STEPS,
            RUN_TRAINING,
            None if eval_metrics is None else round(eval_metrics.get("eval_loss", 0), 4),
        ],
    }
)
display(final_summary)


,score
rouge1,0.2
rouge2,0.2
rougeL,0.2
rougeLsum,0.2


,item,value
0,selected_domain,law
1,train_samples,2000
2,valid_samples,400
3,model_name,digit82/kobart-summarization
4,max_input_length,512
5,max_target_length,144
6,train_epochs,3
7,gradient_accumulation_steps,2
8,run_training,True
9,eval_loss,1.1226


## 10. 결론
- 이번 실험에서는 AI HUB 문서요약 데이터 중 `law` 도메인을 선택하고, 학습 데이터 2,000개와 검증 데이터 400개를 사용하여 `digit82/kobart-summarization` 모델을 파인튜닝하였다. 학습은 3 epoch로 진행했으며, 최종 검증 손실(`eval_loss`)은 약 `1.1226`으로 확인되었다.
- 정성 평가 결과, 모델은 법률 문서의 핵심 쟁점과 결론을 일정 수준 따라가며 요약을 생성할 수 있었고, ROUGE 역시 소규모 샘플 기준 `0.2` 수준으로 확인되어 기본적인 요약 성능은 확보했다고 볼 수 있다. 특히 일부 사례에서는 정답 요약과 유사한 핵심 판시를 잘 반영하였다.
- 반면 생성 문장이 다소 길고 추출형 경향이 남아 있으며, 일부 샘플에서는 표현이 반복되거나 문장이 매끄럽지 않은 한계도 확인되었다. 따라서 이후에는 더 다양한 샘플을 사용한 학습, 학습 epoch 및 생성 파라미터 재조정, 그리고 전체 검증셋 기준의 정량 평가를 추가하면 성능을 더 개선할 수 있을 것으로 판단된다.

## 11. 아쉬운 점 및 향후 개선 방향
- 이번 실험은 제한된 시간 안에서 `law` 도메인만 사용해 요약 파이프라인을 완성하고 성능을 확인하는 데 초점을 두었기 때문에, 다른 도메인에 대해서도 비슷한 성능이 나오는지는 확인하지 못했다.
- 정량 평가는 현재 설정 기준으로 확인했지만, 이후에는 전체 검증셋 또는 더 큰 평가 샘플을 대상으로 ROUGE를 계산해 모델 성능을 더 객관적으로 비교할 필요가 있다.
- 생성 결과는 핵심 내용을 어느 정도 따라가지만, 여전히 문장이 길어지거나 원문 표현을 그대로 가져오는 추출형 경향이 남아 있었다. 따라서 추가 학습과 생성 파라미터 조정을 통해 더 간결하고 자연스러운 요약을 만드는 방향으로 개선할 수 있다.
- 앞으로는 더 큰 데이터 규모를 활용한 학습, 다른 한국어 요약 모델과의 비교 실험, 그리고 도메인별 성능 차이 분석까지 확장하면 더 설득력 있는 결과를 만들 수 있을 것으로 보인다.